# 37 — ReWOO Agent

ReWOO (Reasoning WithOut Observation) emits all tool calls upfront as a structured plan, executes them in bulk with variable substitution, then synthesizes using accumulated evidence — no interleaved observe-think-act loop.

**What you'll learn**
- Why upfront planning beats interleaved ReAct for well-defined tasks
- Structured JSON plan: `[{"tool", "args", "output_var"}, ...]`
- Variable substitution: `$E1` in later steps resolved to prior output
- Planner → Executor → Solver: clean 3-node DAG
- How to strip markdown code fences from LLM JSON output

**Contrast:** `4-basic-react-agent` re-reasons after every observation; ReWOO commits upfront, saving one LLM call per tool step.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langgraph python-dotenv pydantic

In [ ]:
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. ReWOO vs ReAct ─────────────────────────────────────────────────────────
#
# ReAct (example 4):                   ReWOO (this example):
#   think                                plan ALL tool calls at once
#   act → tool call 1                    execute tool 1 ($E1)
#   observe output                       execute tool 2 (uses $E1)
#   think (with observation)             execute tool 3 (uses $E2)
#   act → tool call 2                    solve with all evidence
#   observe                              → final answer
#   final answer
#
# ReAct cost: 1 LLM call per (think+act) pair = N calls for N tools
# ReWOO cost: 1 planner + N executor calls (no reasoning overhead per step)
#
# Variable substitution:
#   plan[0]: {"tool": "search", "args": "X",    "output_var": "$E1"}
#   plan[1]: {"tool": "summarize", "args": "$E1", "output_var": "$E2"}
#   Executor replaces "$E1" in plan[1].args with plan[0]'s output.

print("ReWOO: plan upfront, execute in bulk, solve once.")
print("ReAct: interleave reasoning and observation per tool call.")

In [ ]:
# ── 4. Build the ReWOO graph ──────────────────────────────────────────────────
import json
from typing import TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

PLANNER_PROMPT = """You are a ReWOO planner. Given a task, output a plan as a JSON list.
Each item: {{"tool": "search|summarize|analyze", "args": "...", "output_var": "$E1"}}
Use $E1, $E2, etc. as variable names. Reference prior vars in args.
Return JSON only (no markdown fences).

Task: {task}"""

TOOL_DESCRIPTIONS = {
    "search": "Retrieve information about a topic",
    "summarize": "Condense text into key points",
    "analyze": "Break down and interpret information",
}


class ReWOOState(TypedDict):
    task: str
    plans: list[dict]
    evidence: dict
    final_answer: str


def planner(state: ReWOOState) -> dict:
    result = llm.invoke([HumanMessage(content=PLANNER_PROMPT.format(task=state["task"]))])
    text = result.content.strip()
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:-1])
    try:
        plans = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        plans = []
    print(f"  Plan: {len(plans)} steps")
    for i, p in enumerate(plans):
        print(f"    {i + 1}. {p.get('tool')}({p.get('args', '')[:40]}) → {p.get('output_var')}")
    return {"plans": plans, "evidence": {}}


def executor(state: ReWOOState) -> dict:
    evidence = {}
    for step in state["plans"]:
        tool = step.get("tool", "analyze")
        raw_args = step.get("args", "")
        args = raw_args
        for var, val in evidence.items():
            args = args.replace(var, val[:200])
        desc = TOOL_DESCRIPTIONS.get(tool, "process")
        prompt = (
            f"Tool: {tool} ({desc})\nInput: {args}\nProvide a detailed result in 3-5 sentences."
        )
        result = llm.invoke([HumanMessage(content=prompt)])
        output_var = step.get("output_var", f"$E{len(evidence) + 1}")
        evidence[output_var] = result.content
        print(f"  {output_var}: {result.content[:80]}...")
    return {"evidence": evidence}


def solver(state: ReWOOState) -> dict:
    ev = "\n".join(f"{k}: {v}" for k, v in state["evidence"].items())
    prompt = f"Task: {state['task']}\n\nEvidence:\n{ev}\n\nFinal answer:"
    result = llm.invoke([HumanMessage(content=prompt)])
    return {"final_answer": result.content}


graph = StateGraph(ReWOOState)
graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("solver", solver)
graph.set_entry_point("planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "solver")
graph.add_edge("solver", END)
app = graph.compile()
print("Graph compiled.")

In [ ]:
# ── 5. Run the ReWOO agent ────────────────────────────────────────────────────
TASK = "Research the key differences between LangGraph and LangChain, then summarize in 3 bullet points."

print(f"TASK: {TASK}\n")
result = app.invoke({"task": TASK, "plans": [], "evidence": {}, "final_answer": ""})

print(f"\n{'=' * 60}")
print(f"EVIDENCE VARS: {list(result['evidence'].keys())}")
print(f"\nFINAL ANSWER:\n{result['final_answer']}")

## Exercises

**Exercise 1 — Inspect the plan:** Print the full plan dict after `planner` runs. How many steps did the LLM emit? Are variable references used correctly?

**Exercise 2 — Add real tools:** Replace the simulated executor with `DuckDuckGoSearchRun` for `tool=="search"`. Does plan quality improve?

**Exercise 3 — Parallel execution:** Which steps have no `$E` dependencies and could run concurrently? Implement parallel execution with `asyncio.gather`.

**Exercise 4 — Compare with ReAct:** Run the same task through `4-basic-react-agent`. Count total LLM calls for each. Which is cheaper? Which gives better answers?

## What's next

- **35-plan-execute-agent** — similar upfront planning, executor loops per step
- **4-basic-react-agent** — interleaved observe-think-act for adaptive tasks
- **38-langgraph-command-pattern** — `Command(goto=...)` for edgeless dynamic routing